# Croatian Legal System — Institutional Authority Analysis

**Research question:** Which Croatian institutions hold the most structural authority
in the legal system — and does raw legislative volume correspond to genuine legal influence?

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────────
import json
import math
from collections import defaultdict
from pathlib import Path
from IPython.display import display, Markdown

import altair as alt
import networkx as nx
import pandas as pd

alt.data_transformers.disable_max_rows()

# Load the network from disk. The file is in NetworkX node-link format.
_path = next(
    (p for p in [Path("../network_institutional.json"), Path("network_institutional.json")]
     if p.exists()),
    None,
)
if _path is None:
    raise FileNotFoundError("network_institutional.json not found — run build_institutional_network.py first.")

with open(_path, encoding="utf-8") as _f:
    _raw = json.load(_f)
if "edges" not in _raw and "links" in _raw:
    _raw["edges"] = _raw["links"]
G = nx.node_link_graph(_raw, directed=True, multigraph=False)

# Separate the two node types and build a fast act→institution lookup
act_nodes  = {n: d for n, d in G.nodes(data=True) if d.get("node_type") == "act"}
inst_nodes = {n: d for n, d in G.nodes(data=True) if d.get("node_type") == "institution"}

eli_to_inst: dict = {}
for _s, _t, _ed in G.edges(data=True):
    if _ed.get("relation") == "passed_by":
        eli_to_inst[_s] = _t

inst_name: dict = {iri: d.get("name", iri) for iri, d in inst_nodes.items()}

# Count edges by relation type
relation_counts: dict = defaultdict(int)
for _s, _t, _ed in G.edges(data=True):
    relation_counts[_ed.get("relation", "?")] += 1

In [2]:
# ── Build the act-to-act subgraph ──────────────────────────────────────────────
#
# The full network is heterogeneous (acts + institutions). For centrality
# analysis we need a pure act-to-act graph so that institution nodes do not
# distort scores. We also:
#   - remap 'changes' → 'amends'  (every changes edge has a duplicate amends edge;
#     keeping both would double-count the same relationship)
#   - exclude repealed acts as sources  (a revoked act should not propagate
#     authority forward in time)
#   - drop self-loops

_ACCEPT = {"amends", "based_on", "repeals", "changes"}

repealed_acts = {
    _t for _s, _t, _ed in G.edges(data=True)
    if _ed.get("relation") == "repeals"
}

_G_acts = nx.DiGraph()
_G_acts.add_nodes_from((n, d) for n, d in G.nodes(data=True) if d.get("node_type") == "act")
_G_acts.add_edges_from(
    (_s, _t, {**_ed, "relation": "amends" if _ed["relation"] == "changes" else _ed["relation"]})
    for _s, _t, _ed in G.edges(data=True)
    if _ed.get("relation") in _ACCEPT and _s not in repealed_acts and _s != _t
)

# Work only on the weakly-connected giant component (12,525 of 70,975 act nodes; 17.6%).
# The remaining 82% fall outside the giant: ~44,126 are complete singletons — acts with
# no citation links at all (only a passed_by edge to their institution) — and ~14,300
# are in small isolated clusters. Singletons carry no network signal; keeping them would
# dilute centrality scores without adding information.
_wcc = sorted(nx.weakly_connected_components(_G_acts), key=len, reverse=True)
G_acts_giant_di = _G_acts.subgraph(_wcc[0]).copy()
G_acts_giant    = G_acts_giant_di.to_undirected()

# A based_on-only directed subgraph is used for both PageRank and HITS.
# based_on = "this act formally derives legal authority from that act" —
# the purest signal of the delegation hierarchy.
# All 12,525 giant-component nodes are included; acts with no based_on edges
# become dangling nodes whose score mass is redistributed uniformly by
# NetworkX's default PageRank teleportation — full coverage at the cost of
# a marginal background floor. See the Cell 3 coverage note for the rationale.
G_bo_di = nx.DiGraph(
    (_s, _t) for _s, _t, _ed in G_acts_giant_di.edges(data=True)
    if _ed.get("relation") == "based_on"
)
G_bo_di.add_nodes_from(G_acts_giant_di.nodes(data=True))

# A no-repeals variant is used for PageRank comparison.
G_acts_no_rep_di = nx.DiGraph()
G_acts_no_rep_di.add_nodes_from(G_acts_giant_di.nodes(data=True))
G_acts_no_rep_di.add_edges_from(
    (_s, _t, _ed) for _s, _t, _ed in G_acts_giant_di.edges(data=True)
    if _ed.get("relation") != "repeals"
)

print("Setup complete.")
print(f"  Full network     : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"  Act-to-act giant : {G_acts_giant_di.number_of_nodes():,} nodes, {G_acts_giant_di.number_of_edges():,} edges")
print(f"  G_bo_di          : {G_bo_di.number_of_nodes():,} nodes, {G_bo_di.number_of_edges():,} edges")

# ── Full-network based_on graph (honest representation) ────────────────────────
#
# The giant-component graphs above are used for PageRank and HITS, which need a
# connected graph to produce meaningful scores.  For the Spearman volume-vs-authority
# comparison in Cell 6, however, restricting to the giant creates a population
# mismatch: volume counts all 70,975 acts while authority covers only the 17.6%
# that land in the citation giant.  Institutions whose acts are mostly
# administrative singletons (Vlada's uredbe, appointment decisions, etc.) are
# penalised on volume but invisible on authority — not because they lack authority,
# but because their acts happen not to cite each other.
#
# Fix: build a separate based_on graph that includes every act node.
#   - Acts with no incoming citations score 0.  They are NOT excluded; they appear
#     in both the volume count and the authority count with the same weight of zero.
#     This is the honest representation: an act nobody cites has zero network authority.
#   - The same repealed-as-source filter is applied for consistency: a defunct act's
#     outgoing citations should not contribute to another act's authority count.
#   - Self-loops are dropped as usual.
# Result: G_bo_full is the full-population authority graph.  Cell 6 uses it for the
# primary Spearman comparison; PageRank and HITS on the giant are kept as structural
# characterisations of the citation hierarchy, not as the hypothesis-test metric.
G_bo_full = nx.DiGraph()
G_bo_full.add_nodes_from(
    (n, d) for n, d in G.nodes(data=True) if d.get("node_type") == "act"
)
G_bo_full.add_edges_from(
    (s, t)
    for s, t, d in G.edges(data=True)
    if d.get("relation") == "based_on"
    and s != t
    and s not in repealed_acts   # same filter as the giant analysis
)
_cit_cited   = sum(1 for _, deg in G_bo_full.in_degree() if deg > 0)
_cit_uncited = G_bo_full.number_of_nodes() - _cit_cited
print(f"  G_bo_full        : {G_bo_full.number_of_nodes():,} nodes, "
      f"{G_bo_full.number_of_edges():,} edges")
print(f"  Acts cited >= 1  : {_cit_cited:,}  ({_cit_cited/G_bo_full.number_of_nodes():.1%})")
print(f"  Acts with 0 citations (score = 0): "
      f"{_cit_uncited:,}  ({_cit_uncited/G_bo_full.number_of_nodes():.1%})")

# Quantify citation mass flowing to stub acts.
# Stubs have no passed_by edges, so their in-degree in G_bo_full is never
# counted toward any institution's citation total.  This measures what fraction
# of the 10,021 based_on edges are "unattributed" — they reach a stub whose
# institution is unknown and are dropped during institution-level aggregation.
_acts_with_inst = {
    _s for _s, _t, _ed in G.edges(data=True)
    if _ed.get("relation") == "passed_by"
}
_stubs_in_full = [n for n in G_bo_full.nodes() if n not in _acts_with_inst]
_stub_cit_mass = sum(G_bo_full.in_degree(n) for n in _stubs_in_full)
print(f"  Stub acts in G_bo_full            : {len(_stubs_in_full):,}")
print(f"  Citations to stubs (unattributed) : {_stub_cit_mass:,} of "
      f"{G_bo_full.number_of_edges():,} edges "
      f"({_stub_cit_mass / max(G_bo_full.number_of_edges(), 1):.1%})")


---
## Cell 1 — Data Summary

Before any analysis: what data are we working with, where does it come from,
and what are the basic properties of the network?

In [3]:
# ── Data summary ───────────────────────────────────────────────────────────────
#
# This cell shows the raw shape of the dataset so the reader can assess scale
# and understand how the data was assembled before seeing any results.

_Gu       = G.to_undirected()
_comps    = sorted(nx.connected_components(_Gu), key=len, reverse=True)
_giant_pct = len(_comps[0]) / G.number_of_nodes()

inferred_acts = {n for n, d in act_nodes.items() if d.get("institution_inferred")}
curated_acts  = {n for n, d in act_nodes.items() if d.get("manual_curated")}
stub_nodes    = {n for n, d in G.nodes(data=True) if d.get("stub")}
explicit_acts = len(eli_to_inst) - len(inferred_acts) - len(curated_acts)

display(Markdown("### Network overview"))
display(pd.DataFrame([
    ["Total nodes",          f"{G.number_of_nodes():,}"],
    ["  — act nodes",        f"{len(act_nodes):,}"],
    ["  — institution nodes",f"{len(inst_nodes)}"],
    ["Total edges",          f"{G.number_of_edges():,}"],
    ["Giant component",      f"{len(_comps[0]):,} nodes ({_giant_pct:.1%})"],
    ["Components",           f"{len(_comps):,}"],
], columns=["Metric", "Value"]))

display(Markdown("### How each act was attributed to an institution"))
display(pd.DataFrame([
    ["Explicit",         f"{explicit_acts:,}", "passed_by_iri present in JSON-LD scraped from Narodne novine"],
    ["Inferred",         f"{len(inferred_acts):,}", "passed_by_iri missing — institution inferred from document type (ZAKON→Sabor, UREDBA→Vlada)"],
    ["Manually curated", f"{len(curated_acts)}",  "3 high-in-degree acts verified by hand from the Narodne novine HTML page"],
    ["Unattributed stubs",f"{len(stub_nodes):,}", "Referenced in citation links but absent from the DB — institution unknown"],
], columns=["Category", "Acts", "Source"]))

display(Markdown("### Edge types in the network"))
display(pd.DataFrame([
    ["passed_by", f"{relation_counts['passed_by']:,}", "act → institution", "Administrative authorship — grouping key only, not used for centrality"],
    ["amends",    f"{relation_counts['amends']:,}",    "newer → older",     "Modifies specific articles of the target act"],
    ["based_on",  f"{relation_counts['based_on']:,}",  "newer → older",     "Formally derives legal authority from the target act — primary authority signal"],
    ["changes",   f"{relation_counts['changes']:,}",   "newer → older",     "General substantive change — strict subset of amends, remapped at analysis time"],
    ["repeals",   f"{relation_counts['repeals']:,}",   "newer → older",     "Revokes the target act entirely"],
    ["corrects",  f"{relation_counts['corrects']:,}",  "any",               "Gazette erratum — excluded from analysis"],
], columns=["Relation", "Count", "Direction", "Meaning"]))

display(Markdown("### Top 10 institutions by legislative volume"))
display(pd.DataFrame([
    {"Institution": inst_name[iri], "Acts": d["act_count"]}
    for iri, d in sorted(inst_nodes.items(), key=lambda x: -x[1]["act_count"])[:10]
]))

### Network overview

,Metric,Value
0,Total nodes,"71,021"
1,— act nodes,"70,975"
2,— institution nodes,46
3,Total edges,"90,588"
4,Giant component,"65,995 nodes (92.9%)"
5,Components,"1,627"


### How each act was attributed to an institution

,Category,Acts,Source
0,Explicit,"59,249",passed_by_iri present in JSON-LD scraped from ...
1,Inferred,"2,099",passed_by_iri missing — institution inferred f...
2,Manually curated,3,3 high-in-degree acts verified by hand from th...
3,Unattributed stubs,"6,393",Referenced in citation links but absent from t...


### Edge types in the network

,Relation,Count,Direction,Meaning
0,passed_by,"61,351",act → institution,"Administrative authorship — grouping key only,..."
1,amends,"6,534",newer → older,Modifies specific articles of the target act
2,based_on,"11,727",newer → older,Formally derives legal authority from the targ...
3,changes,"5,778",newer → older,General substantive change — strict subset of ...
4,repeals,"4,335",newer → older,Revokes the target act entirely
5,corrects,863,any,Gazette erratum — excluded from analysis


### Top 10 institutions by legislative volume

,Institution,Acts
0,Vlada RH,24608
1,Hrvatski sabor,9174
2,Predsjedništvo RH,5577
3,Min. poljoprivrede,3822
4,HANFA,2642
5,Min. zdravstva,1229
6,Predsjednik RH,1219
7,HZMO,1185
8,HNB,883
9,Visoki upravni sud RH,849


---
## Cell 1b — Network Characterization

Before computing authority scores, we need to characterise the structural type
of this network. This determines which analytical tools are appropriate and sets
expectations for the results.

**Key questions this cell answers:**

- **Sparsity** — density and average degree tell us whether this is a dense citation
  graph or a sparse one (most legislative citation networks are very sparse).
- **Power-law in-degree?** — if the log-log degree distribution is linear, a small
  number of foundational laws attract most citations. PageRank and HITS are
  especially informative in such networks.
- **Reciprocity** — citation edges point chronologically backward (newer → older),
  so reciprocity should be near zero. A non-negligible reciprocity would flag data
  quality issues (circular references).
- **Assortativity** — negative (disassortative) means high-degree foundational
  laws are cited by many low-degree implementing acts: a hub-and-spoke structure
  that justifies the HITS authority/hub split in Cell 4.
- **Small-world?** — low average path length relative to network size, combined
  with non-trivial clustering, is the signature of a small-world network.
- **Non-giant components** — characterising the excluded components confirms
  whether they are truly isolated acts or suppressed signal.

In [4]:
# ── Network characterization ─────────────────────────────────────────────────
import random
from collections import Counter

# 1. Global topology
_n           = G_acts_giant_di.number_of_nodes()
_m           = G_acts_giant_di.number_of_edges()
_density     = nx.density(G_acts_giant_di)
_reciprocity = nx.reciprocity(G_acts_giant_di)
_avg_clust   = nx.average_clustering(G_acts_giant)
_assort      = nx.degree_assortativity_coefficient(G_acts_giant)

# 2. Degree statistics (directed)
_in_deg  = dict(G_acts_giant_di.in_degree())
_out_deg = dict(G_acts_giant_di.out_degree())
_max_in  = max(_in_deg.values())
_max_out = max(_out_deg.values())

# 3. Approx. average path length — BFS sample of 500 source nodes (fast, unbiased)
random.seed(42)
_sample_nodes = random.sample(list(G_acts_giant.nodes()), 500)
_pl_total, _pl_pairs = 0, 0
for _src in _sample_nodes:
    _lengths = nx.single_source_shortest_path_length(G_acts_giant, _src)
    _pl_total += sum(_lengths.values())
    _pl_pairs += len(_lengths) - 1   # exclude the self-distance of 0
_avg_pl = _pl_total / _pl_pairs

# 4. Diameter lower bound via double-sweep BFS (O(2*(n+m)) — runs in < 1 s)
# Two BFS traversals: start from an arbitrary node, jump to the farthest node,
# BFS again — the resulting distance is a tight lower bound on the true diameter.
_s0    = next(iter(G_acts_giant.nodes()))
_d0    = nx.single_source_shortest_path_length(G_acts_giant, _s0)
_far1  = max(_d0, key=_d0.get)
_d1    = nx.single_source_shortest_path_length(G_acts_giant, _far1)
_diam_lb = _d1[max(_d1, key=_d1.get)]

# 5. Non-giant component breakdown (_wcc was built from _G_acts earlier)
_nongiant_sizes = [len(c) for c in _wcc[1:]]
_n_singletons   = sum(1 for s in _nongiant_sizes if s == 1)
_n_nontrivial   = sum(1 for s in _nongiant_sizes if s > 1)
_nongiant_total = sum(_nongiant_sizes)

display(Markdown("### Structural metrics — act-to-act giant component"))
display(pd.DataFrame([
    ["Nodes",                                  f"{_n:,}"],
    ["Edges",                                  f"{_m:,}"],
    ["Density",                                f"{_density:.2e}"],
    ["Avg undirected degree  2m/n",            f"{2 * _m / _n:.3f}"],
    ["Max in-degree",                          f"{_max_in}"],
    ["Max out-degree",                         f"{_max_out}"],
    ["Reciprocity",                            f"{_reciprocity:.4f}"],
    ["Avg clustering coefficient",             f"{_avg_clust:.4f}"],
    ["Degree assortativity coefficient",       f"{_assort:.4f}"],
    ["Approx. avg path length (500-node BFS)", f"{_avg_pl:.2f}"],
    ["Diameter lower bound (double-sweep)",    f"≥ {_diam_lb}"],
    ["Non-giant components",
     f"{len(_nongiant_sizes):,}  total nodes: {_nongiant_total:,}"
     f"  ({_n_singletons:,} singletons, {_n_nontrivial} non-trivial)"],
], columns=["Metric", "Value"]))

# ── Degree distribution — log-log ──────────────────────────────────────────────
# A straight line on a log-log plot indicates power-law behaviour.
# In-degree follows power law in citation networks; out-degree is typically narrower.
_in_cnt  = Counter(v for v in _in_deg.values()  if v > 0)
_out_cnt = Counter(v for v in _out_deg.values() if v > 0)

_dd_df = pd.concat([
    pd.DataFrame([{"degree": k, "count": v, "Direction": "in-degree"}  for k, v in _in_cnt.items()]),
    pd.DataFrame([{"degree": k, "count": v, "Direction": "out-degree"} for k, v in _out_cnt.items()]),
], ignore_index=True)

_dd_chart = (
    alt.Chart(_dd_df)
    .mark_point(size=40, filled=True, opacity=0.65)
    .encode(
        x=alt.X("degree:Q",
                scale=alt.Scale(type="log"),
                title="Degree k  (log scale)"),
        y=alt.Y("count:Q",
                scale=alt.Scale(type="log"),
                title="Number of nodes  (log scale)"),
        color=alt.Color("Direction:N", title=""),
        tooltip=["Direction", "degree", "count"],
    )
    .properties(
        title="Degree distribution — act-to-act giant  (log-log)",
        width=520, height=360,
    )
)

# ── Non-giant component size distribution ─────────────────────────────────────
_comp_size_df = (
    pd.DataFrame(
        Counter(_nongiant_sizes).items(),
        columns=["Component size", "Components"]
    ).sort_values("Component size")
)

_comp_chart = (
    alt.Chart(_comp_size_df)
    .mark_bar(color="#9ecae1")
    .encode(
        x=alt.X("Component size:O", title="Component size (nodes)"),
        y=alt.Y("Components:Q",     title="Number of components"),
        tooltip=["Component size", "Components"],
    )
    .properties(
        title=f"Non-giant component sizes  ({len(_nongiant_sizes):,} components, {_nongiant_total:,} nodes excluded)",
        width=380, height=300,
    )
)

display(alt.hconcat(_dd_chart, _comp_chart))

### Structural metrics — act-to-act giant component

,Metric,Value
0,Nodes,"12,525"
1,Edges,"15,121"
2,Density,9.64e-05
3,Avg undirected degree 2m/n,2.415
4,Max in-degree,1468
5,Max out-degree,18
6,Reciprocity,0.0000
7,Avg clustering coefficient,0.0134
8,Degree assortativity coefficient,-0.0963
9,Approx. avg path length (500-node BFS),8.48


alt.HConcatChart(...)

---
## Cell 2 — Legislative Volume

The null hypothesis: if volume equals authority, this chart is the complete answer.
Vlada RH produces more than 2.5× as many acts as the next institution.
The rest of the analysis tests whether that lead holds up when we measure
structural influence rather than raw count.

**Note on population scope:** the volume metric counts all acts an institution
published in Narodne novine (70,975 acts total). The authority metrics in Cells 3–4
operate on the 12,525-act citation giant — acts that formally participate in the
legal delegation hierarchy (amending, repealing, or deriving authority from other
acts). The two metrics intentionally use different populations: volume captures
total legislative output, authority captures structural position within the
citation network. Comparing them directly is the point — an institution whose
acts are mostly administrative decisions with no citation links will rank high on
volume but will be absent from or weak in the citation giant.


In [5]:
# ── Legislative volume ─────────────────────────────────────────────────────────
#
# Simple bar chart of act count per institution.
# This is not an analysis result — it is the baseline we will compare against.
# If every subsequent metric ranks institutions in the same order as this chart,
# the answer to the research question is "yes, volume = authority."
# If the order changes, it does not.

_vol_df = pd.DataFrame([
    {"Institution": inst_name[iri], "Acts": d["act_count"]}
    for iri, d in sorted(inst_nodes.items(), key=lambda x: -x[1]["act_count"])[:15]
])

alt.Chart(_vol_df).mark_bar(color="#4c78a8").encode(
    x=alt.X("Acts:Q", title="Number of acts published"),
    y=alt.Y("Institution:N", sort="-x", title=None),
    tooltip=["Institution", "Acts"],
).properties(
    title="Legislative volume — Top 15 institutions",
    width=640, height=400,
)

alt.Chart(...)

---
## Cell 3 — PageRank: Structural Authority

PageRank treats the citation network as a random walk: an act that is cited
by many other well-cited acts receives a high score. It measures how much
legal weight flows through an act.

**Two graph variants are compared to isolate what drives each institution's score:**

- **Full graph** (amends + based_on + repeals): broad importance in the citation
  network. Conflates three different signals — authority, longevity, and obsolescence.
- **based_on only**: only formal delegation links are counted. An act scores high
  only if many other acts explicitly cite it as their legal basis. Runs on the full
  G_bo_di (12,525 nodes) so that no delegation subgraph is silently excluded —
  see the coverage note below.

For each variant, **average PageRank per act** is the volume-neutral metric —
total score divided by the number of acts the institution has in the giant.
If Sabor's average is higher than Vlada's despite having fewer acts, it means
Sabor's laws are individually more foundational.

#### Coverage note — why G\_bo\_di instead of the WCC

The original code restricted the based\_on PageRank graph to the largest weakly-connected component of G\_bo\_di (`G_bo_pr`, 6,130 nodes) to prevent dangling-node teleportation from inflating scores. That choice introduced a more serious problem: **49 % of act nodes were excluded**, including 5,727 non-trivial components — the second-largest alone has 92 nodes connected by real based\_on edges. Those delegation chains are likely isolated because they reference specialised legal bases (EU-transposition acts, military law, international treaties) that are not linked to the main civilian hierarchy. Silently dropping them means any foundational act in those subgraphs scores zero, and any institution whose acts are concentrated there would be entirely absent from the authority ranking.

**The fix:** run PageRank on the full G\_bo\_di (all 12,525 nodes). NetworkX's default handling redistributes dangling-node probability mass uniformly across all nodes via the teleportation term `(1 − α)/N`. This adds a background floor of roughly `0.15 / 12525 ≈ 1.2 × 10⁻⁵` to every node's score — negligible relative to the scores of genuinely authoritative acts (which accumulate via citations). The main Sabor/Vlada finding is unaffected; the gain is that no delegation subgraph is arbitrarily excluded.

In [6]:

# ── PageRank ───────────────────────────────────────────────────────────────────
#
# Both variants run on the directed giant component.
# passed_by edges are excluded in both cases — mixing institution nodes into
# the graph would reward institutions just for having many acts in the network.
# Results are aggregated to institution level after the fact.
#
# alpha=0.85 is the standard damping factor.
# weight=None — all edges treated equally (no meaningful edge weights exist).

print("Computing PageRank — full graph (amends + based_on + repeals)…")
_pr_full = nx.pagerank(G_acts_giant_di, alpha=0.85, max_iter=300, weight=None)

print("Computing PageRank — based_on only (full G_bo_di, 12,525 nodes)…")
_pr_bo = nx.pagerank(G_bo_di, alpha=0.85, max_iter=300, weight=None)

def _agg_pr(pr_dict):
    # Sum and count PageRank scores per institution.
    tot: dict = defaultdict(float)
    cnt: dict = defaultdict(int)
    for eli, sc in pr_dict.items():
        i = eli_to_inst.get(eli)
        if i in inst_nodes:
            tot[i] += sc
            cnt[i] += 1
    return tot, cnt

_tot_full, _cnt_full = _agg_pr(_pr_full)
_tot_bo,   _cnt_bo   = _agg_pr(_pr_bo)

_pr_rows = []
for iri in inst_nodes:
    n_full = _cnt_full.get(iri, 0)
    n_bo   = _cnt_bo.get(iri, 0)
    if n_full < 10:
        continue
    _pr_rows.append({
        "Institution":   inst_name[iri],
        "avg_pr_full":   round(_tot_full[iri] / n_full, 8),
        "avg_pr_bo":     round(_tot_bo.get(iri, 0) / n_bo, 8) if n_bo > 0 else 0.0,
        "acts_in_giant": n_full,
        "total_acts":    inst_nodes[iri]["act_count"],
    })

_df_pr = pd.DataFrame(_pr_rows)

# volume_rank = absolute rank by total published acts across ALL institutions.
# Using total acts (not acts_in_giant) makes this directly comparable to the
# Cell 2 volume bar chart and to the exclusion table (Predsjedništvo = rank 3).
_all_vol_ranks = {inst_name[iri]: r + 1 for r, (iri, _) in enumerate(
    sorted(inst_nodes.items(), key=lambda x: -x[1]["act_count"])
)}
_df_pr["volume_rank"]   = _df_pr["Institution"].map(_all_vol_ranks)
_df_pr["avg_full_rank"] = _df_pr["avg_pr_full"].rank(ascending=False).astype(int)
_df_pr["avg_bo_rank"]   = _df_pr["avg_pr_bo"].rank(ascending=False).astype(int)


Computing PageRank — full graph (amends + based_on + repeals)…


Computing PageRank — based_on only (full G_bo_di, 12,525 nodes)…


In [7]:
# Side-by-side comparison of the two PageRank variants.
# If the rankings differ significantly, it means amends and repeals edges
# were distorting the full-graph scores — the based_on-only variant strips
# that noise and shows delegation authority more cleanly.

_chart_full = (
    alt.Chart(_df_pr.sort_values("avg_pr_full", ascending=False).head(15))
    .mark_bar(color="#f58518")
    .encode(
        x=alt.X("avg_pr_full:Q", title="Avg PageRank per act"),
        y=alt.Y("Institution:N", sort="-x", title=None),
        tooltip=["Institution", "avg_pr_full", "acts_in_giant"],
    )
    .properties(title="Full graph (amends + based_on + repeals)", width=380, height=380)
)

_chart_bo = (
    alt.Chart(_df_pr.sort_values("avg_pr_bo", ascending=False).head(15))
    .mark_bar(color="#e45756")
    .encode(
        x=alt.X("avg_pr_bo:Q", title="Avg PageRank per act"),
        y=alt.Y("Institution:N", sort="-x", title=None),
        tooltip=["Institution", "avg_pr_bo", "acts_in_giant"],
    )
    .properties(title="based_on only — pure delegation authority", width=380, height=380)
)

display(Markdown("**Average PageRank per act — full graph vs based_on only**"))
display(alt.hconcat(_chart_full, _chart_bo))

**Average PageRank per act — full graph vs based_on only**

alt.HConcatChart(...)

In [8]:
# Rank shift between the two variants.
# Positive shift = institution rises when amends/repeals are removed →
#   its authority comes from genuine delegation, not amendment churn or obsolescence.
# Negative shift = institution drops when amends/repeals are removed →
#   its full-graph score was inflated by non-delegation edges.
_df_pr["rank_shift"] = _df_pr["avg_full_rank"] - _df_pr["avg_bo_rank"]

display(Markdown("**Rank shift: full graph → based_on only** (positive = rises, negative = drops)"))
display(
    alt.Chart(_df_pr.sort_values("avg_bo_rank"))
    .mark_bar()
    .encode(
        x=alt.X("rank_shift:Q", title="Rank shift (full → based_on only)"),
        y=alt.Y("Institution:N", sort=alt.EncodingSortField("avg_bo_rank"), title=None),
        color=alt.condition(
            alt.datum.rank_shift >= 0,
            alt.value("#4daf4a"),
            alt.value("#e41a1c"),
        ),
        tooltip=["Institution", "avg_full_rank", "avg_bo_rank", "rank_shift"],
    )
    .properties(
        title="Rank shift — green = authority driven by delegation, red = inflated by amends/repeals",
        width=560, height=400,
    )
)

**Rank shift: full graph → based_on only** (positive = rises, negative = drops)

alt.Chart(...)

---
## Cell 4 — HITS: Hub vs Authority

PageRank gives a single score. HITS separates two fundamentally different
structural roles that PageRank conflates:

- **Authority**: an act that many hub acts cite as their legal basis.
  High authority = foundational framework law (e.g., Zakon o Vladi RH).
- **Hub**: an act that cites many authoritative acts as its legal basis.
  High hub score = implementing regulation that derives authority from many sources.

HITS is run on the based_on-only subgraph so the hub/authority semantics
are clean — only formal delegation links are counted, not amendments or repeals.

Expected result: Sabor = high authority (its zakoni are cited as legal basis),
Vlada = high hub (its uredbe cite many of Sabor's zakoni).
If confirmed, this directly answers the research question: volume ≠ authority.

In [9]:
# ── HITS ───────────────────────────────────────────────────────────────────────
#
# nx.hits returns (hubs dict, authorities dict) keyed by node.
# We aggregate to institution level and then divide by act count to get
# per-act averages. Using totals would give volume-proportional scores
# (institutions with more acts would always dominate), making HITS
# incomparable to PageRank which already aggregates as per-act averages.

print("Computing HITS…")
# nx.hits() raises PowerIterationFailedConvergence if max_iter is exhausted
# without convergence — the absence of an exception below confirms convergence.
# We also report authority concentration: HITS power iteration can be numerically
# fragile when a single node holds most of the authority mass (Zakon o Vladi RH
# holds ~51% in this graph).  High concentration is expected here — it reflects
# the actual topology — but is worth flagging explicitly so the reader knows the
# scores are not an artefact of premature termination.
hubs_r, auth_r = nx.hits(G_bo_di, max_iter=300, normalized=True)
_top_auth_share = max(auth_r.values()) / sum(auth_r.values())
print(f"HITS converged within max_iter=300 (no PowerIterationFailedConvergence).")
print(f"Top authority node holds {_top_auth_share:.1%} of total authority mass "
      f"({'extreme concentration — scores valid but distribution is near-degenerate' if _top_auth_share > 0.40 else 'normal'}).")

_i_auth: dict = defaultdict(float)
_i_hub:  dict = defaultdict(float)
_i_cnt:  dict = defaultdict(int)
for eli, sc in auth_r.items():
    i = eli_to_inst.get(eli)
    if i in inst_nodes:
        _i_auth[i] += sc
        _i_cnt[i]  += 1
for eli, sc in hubs_r.items():
    i = eli_to_inst.get(eli)
    if i in inst_nodes:
        _i_hub[i] += sc

_hits_rows = [
    {
        "Institution":   inst_name[iri],
        "Avg Authority": round(_i_auth[iri] / _i_cnt[iri], 8),
        "Avg Hub":       round(_i_hub.get(iri, 0) / _i_cnt[iri], 8),
        "Acts":          _i_cnt.get(iri, 0),
    }
    for iri in inst_nodes
    if _i_cnt.get(iri, 0) >= 10
]
_df_hits = pd.DataFrame(_hits_rows)

In [10]:
# Side-by-side authority and hub bar charts.
# If Sabor ranks high on authority and Vlada ranks high on hub,
# the data confirms that the two institutions play structurally opposite roles —
# one generates the legal foundations, the other implements them.

_auth_chart = (
    alt.Chart(_df_hits.sort_values("Avg Authority", ascending=False).head(12))
    .mark_bar(color="#e45756")
    .encode(
        x=alt.X("Avg Authority:Q", title="Avg authority score per act"),
        y=alt.Y("Institution:N", sort="-x", title=None),
        tooltip=["Institution", "Avg Authority", "Acts"],
    )
    .properties(title="HITS Authority per act — cited as legal foundation", width=400, height=340)
)

_hub_chart = (
    alt.Chart(_df_hits.sort_values("Avg Hub", ascending=False).head(12))
    .mark_bar(color="#4c78a8")
    .encode(
        x=alt.X("Avg Hub:Q", title="Avg hub score per act"),
        y=alt.Y("Institution:N", sort="-x", title=None),
        tooltip=["Institution", "Avg Hub", "Acts"],
    )
    .properties(title="HITS Hub per act — cites many authoritative sources", width=400, height=340)
)

display(alt.hconcat(_auth_chart, _hub_chart))

alt.HConcatChart(...)

In [11]:

# The top individual acts behind the scores — shows concretely which laws
# are driving an institution's authority or hub position.
# Stub nodes (no title/year/type in DB) are flagged explicitly; they appear
# because other acts cited them, but their metadata was never scraped.

_top_n = 8

_top_auth = pd.DataFrame([
    {
        "Title": (G_bo_di.nodes[eli].get("title") or f"[stub] {eli}")[:70],
        "Year":  G_bo_di.nodes[eli].get("year"),
        "Type":  G_bo_di.nodes[eli].get("document_type"),
        "Stub":  G_bo_di.nodes[eli].get("stub", G_bo_di.nodes[eli].get("title") is None),
        "Authority score": round(sc, 8),
    }
    for eli, sc in sorted(auth_r.items(), key=lambda x: -x[1])[:_top_n]
])

_top_hub = pd.DataFrame([
    {
        "Title": (G_bo_di.nodes[eli].get("title") or f"[stub] {eli}")[:70],
        "Year":  G_bo_di.nodes[eli].get("year"),
        "Type":  G_bo_di.nodes[eli].get("document_type"),
        "Hub score": round(sc, 8),
    }
    for eli, sc in sorted(hubs_r.items(), key=lambda x: -x[1])[:_top_n]
])

display(Markdown(f"**Top {_top_n} acts by authority score** — the structural foundations of Croatian law"))
display(_top_auth)
display(Markdown(f"**Top {_top_n} acts by hub score** — acts that formally derive authority from many sources"))
display(_top_hub)


**Top 8 acts by authority score** — the structural foundations of Croatian law

,Title,Year,Type,Stub,Authority score
0,Zakon o Vladi Republike Hrvatske,2011.0,ZAKON,False,0.514259
1,Zakon o sustavu državne uprave,2011.0,ZAKON,False,0.167699
2,Zakon o zdravstvenoj zaštiti,2008.0,ZAKON,False,0.039798
3,Zakon o lokalnoj i područnoj (regionalnoj) sam...,2001.0,ZAKON,False,0.015112
4,Zakon o državnim službenicima,2005.0,ZAKON,False,0.012564
5,Zakon o pomorskom dobru i morskim lukama,2003.0,ZAKON,False,0.008751
6,[stub] https://narodne-novine.nn.hr/eli/sluzbe...,NaN,NaN,True,0.006570
7,Zakon o poticanju razvoja malog gospodarstva,2002.0,ZAKON,False,0.006500


**Top 8 acts by hub score** — acts that formally derive authority from many sources

,Title,Year,Type,Hub score
0,Rješenje o razrješenju glavnog tajnika Minista...,2015,RJESENJE,0.000826
1,Rješenje o imenovanju zamjenika ravnatelja Drž...,2015,RJESENJE,0.000826
2,Rješenje o imenovanju glavne tajnice Ministars...,2017,RJESENJE,0.000826
3,Rješenje o razrješenju glavnog tajnika Minista...,2016,RJESENJE,0.000826
4,Rješenje o razrješenju glavne tajnice Ministar...,2016,RJESENJE,0.000826
5,Rješenje o imenovanju glavnog tajnika Ministar...,2016,RJESENJE,0.000826
6,Rješenje o imenovanju glavne tajnice Ministars...,2016,RJESENJE,0.000826
7,Rješenje o imenovanju glavnog tajnika Ministar...,2016,RJESENJE,0.000826


#### Interpreting the HITS results

**Why do all 8 top-hub acts have identical scores?**

In HITS, the hub score of a node is proportional to the *sum of authority scores of the nodes it cites*. If two acts cite exactly the same set of authority nodes, they converge to the same hub score during power iteration. The 8 top-scoring hubs are all `RJESENJE` (administrative appointment/dismissal decisions) that cite a single authority source — `Zakon o Vladi RH` — as their formal legal basis. Because their out-neighbourhoods are identical, their hub scores are arithmetically equal. This is *not* a data error; it reflects that these decisions are structurally equivalent in the delegation graph. However, it also reveals a semantic limitation: a pro-forma citation of the Law on Government in every appointment decision is not the same kind of delegation as a complex regulation drawing authority from multiple zakoni. The hub metric cannot distinguish between these two cases.

**Why does a single act capture 51.4% of total HITS authority?**

`Zakon o Vladi RH` (2011) is the primary enabling legislation for virtually all Croatian government activity. Croatian public law requires every Vlada regulation to cite this law as its formal legal basis, creating a near-complete star topology in the `based_on` subgraph with this act at the centre. The extreme concentration (rank 1 authority score: 0.514, rank 2: 0.168) is therefore a genuine structural property of the legal system — not a measurement artefact — but it does mean that HITS authority is heavily dominated by this one node, and the scores of all other acts should be interpreted on a compressed residual scale.

---
## Cell 5 — Summary: Does Volume Equal Authority?

Collects all three rankings (volume, based_on-only average PageRank, HITS authority)
into one table. All authority metrics now use based_on edges only, so they are
directly comparable. A large gap between volume rank and the authority ranks
is the direct answer to the research question.

In [12]:

# ── Summary comparison ─────────────────────────────────────────────────────────
# Builds the final comparison table across all three metrics plus total authority
# share. Institutions below the 10-act threshold are shown in a separate table;
# notably Predsjedništvo RH (volume rank 3, only 7 of 5,577 acts in citation giant),
# the single clearest evidence that legislative volume and structural authority diverge.

# Both PageRank and HITS authority sum to 1.0 after NetworkX normalisation, so
# multiplying by 100 directly gives each institution's % share of global delegation mass.
# max(..., 0) clips floating-point negatives from HITS power iteration (values like -1e-15).
_total_pr_share   = {inst_name[iri]: round(_tot_bo.get(iri, 0) * 100, 3) for iri in inst_nodes}
_total_hits_share = {inst_name[iri]: round(max(_i_auth.get(iri, 0), 0) * 100, 3) for iri in inst_nodes}

_hits_rank = (
    _df_hits[["Institution", "Avg Authority"]]
    .sort_values("Avg Authority", ascending=False)
    .reset_index(drop=True)
)
_hits_rank["hits_authority_rank"] = range(1, len(_hits_rank) + 1)

_df_pr["total_pr_pct"]   = _df_pr["Institution"].map(_total_pr_share).fillna(0.0)
_df_pr["total_hits_pct"] = _df_pr["Institution"].map(_total_hits_share).fillna(0.0)

_summary = (
    _df_pr[["Institution", "volume_rank", "avg_bo_rank", "rank_shift",
            "total_pr_pct", "total_hits_pct"]]
    .merge(_hits_rank[["Institution", "hits_authority_rank"]], on="Institution", how="inner")
    .sort_values("hits_authority_rank")
    .reset_index(drop=True)
)

display(Markdown("""
### Final comparison: volume rank vs structural authority ranks

| Column | Meaning |
|---|---|
| `volume_rank` | Rank by total published acts across all 46 institutions (matches Cell 2 bar chart; Predsjedništvo = rank 3 but below threshold) |
| `avg_bo_rank` | Rank by average PageRank per act, based_on only (1 = highest per-act delegation authority); rank 27 is a shared floor — 18 institutions receive only the teleportation floor (no based_on edges point to their giant-component acts as targets) |
| `rank_shift` | Rank change when amends/repeals edges removed from PageRank (positive = rises when noise stripped) |
| `hits_authority_rank` | Rank by HITS authority per act, based_on only (1 = most cited as legal foundation) |
| `total_pr_pct` | % of total delegation PageRank mass anchored by this institution's giant acts |
| `total_hits_pct` | % of total HITS authority mass anchored by this institution's giant acts |

**Average vs total:** `avg_*_rank` is a per-act efficiency metric (volume-neutral). `total_*_pct`
is the aggregate footprint — which institution collectively dominates the delegation hierarchy.
Institutions with high average authority rank but near-zero total (e.g. Kolektivni ugovori) have
very few acts in the giant; their high average is a division-by-small-number artefact,
not evidence of broad structural authority.
"""))
display(_summary)

# ── Institutions excluded by the 10-act threshold ─────────────────────────────
# Institutions with 0–9 acts in the citation giant are excluded from the authority
# ranking above but still appear in the volume count. Showing them here documents
# the threshold and exposes Predsjedništvo RH as the most analytically important case.

_vol_rank_all = {iri: r + 1 for r, (iri, _) in enumerate(
    sorted(inst_nodes.items(), key=lambda x: -x[1]["act_count"])
)}
_below = []
for iri, d in inst_nodes.items():
    n_giant = _cnt_full.get(iri, 0)
    if n_giant < 10:
        _pct = 100 * n_giant / d["act_count"] if d["act_count"] else 0.0
        _below.append({
            "Institution":   inst_name[iri],
            "volume_rank":   _vol_rank_all[iri],
            "total_acts":    d["act_count"],
            "acts_in_giant": n_giant,
            "note": f"Yugoslav-era body — {n_giant} of {d['act_count']} acts ({_pct:.1f}%) in citation giant"
                    if "Predsjedništvo" in inst_name[iri] else
                    "too few acts in citation giant for stable authority score",
        })

# Compute exact Predsjedništvo stats for the narrative below
_predsj_iri   = next((iri for iri in inst_nodes if "Predsjedništvo" in inst_name[iri]), None)
_predsj_giant = _cnt_full.get(_predsj_iri, 0) if _predsj_iri else 0
_predsj_total = inst_nodes[_predsj_iri]["act_count"] if _predsj_iri else 0
_predsj_pct   = 100 * _predsj_giant / _predsj_total if _predsj_total else 0.0

display(Markdown(f"""
### Institutions excluded from the authority ranking (fewer than 10 acts in citation giant)

**Predsjedništvo RH** is the most analytically significant entry: volume rank 3 with
{_predsj_total:,} acts published, yet only **{_predsj_giant} acts ({_predsj_pct:.1f}%) reach
the citation giant**. It is the collective Presidency of the Socialist Republic of Croatia
(pre-independence), whose acts were issued under the Yugoslav constitutional order and are
structurally isolated from the post-1991 delegation hierarchy. This is the clearest single
data point supporting the research conclusion: raw legislative output and structural
delegation authority are independent dimensions.
"""))
display(pd.DataFrame(_below).sort_values("volume_rank").reset_index(drop=True))



### Final comparison: volume rank vs structural authority ranks

| Column | Meaning |
|---|---|
| `volume_rank` | Rank by total published acts across all 46 institutions (matches Cell 2 bar chart; Predsjedništvo = rank 3 but below threshold) |
| `avg_bo_rank` | Rank by average PageRank per act, based_on only (1 = highest per-act delegation authority); rank 27 is a shared floor — 18 institutions receive only the teleportation floor (no based_on edges point to their giant-component acts as targets) |
| `rank_shift` | Rank change when amends/repeals edges removed from PageRank (positive = rises when noise stripped) |
| `hits_authority_rank` | Rank by HITS authority per act, based_on only (1 = most cited as legal foundation) |
| `total_pr_pct` | % of total delegation PageRank mass anchored by this institution's giant acts |
| `total_hits_pct` | % of total HITS authority mass anchored by this institution's giant acts |

**Average vs total:** `avg_*_rank` is a per-act efficiency metric (volume-neutral). `total_*_pct`
is the aggregate footprint — which institution collectively dominates the delegation hierarchy.
Institutions with high average authority rank but near-zero total (e.g. Kolektivni ugovori) have
very few acts in the giant; their high average is a division-by-small-number artefact,
not evidence of broad structural authority.


,Institution,volume_rank,avg_bo_rank,rank_shift,total_pr_pct,total_hits_pct,hits_authority_rank
0,Hrvatski sabor,2,1,0,35.738,91.214,1
1,Vlada RH,1,8,14,16.682,4.801,2
2,HZZO,11,3,0,1.857,0.440,3
3,Min. zdravstva,6,5,14,1.266,0.016,4
4,Kolektivni ugovori,41,9,12,0.057,0.000,5
5,Min. poljoprivrede,4,11,-3,6.392,0.011,6
6,Min. rada i mir. sustava,33,17,-10,0.482,0.000,7
7,Predsjednik RH,7,2,14,1.110,0.000,8
8,Min. financija,18,14,-4,1.727,0.000,9
9,Min. gospodarstva,46,27,-4,0.195,0.000,10



### Institutions excluded from the authority ranking (fewer than 10 acts in citation giant)

**Predsjedništvo RH** is the most analytically significant entry: volume rank 3 with
5,577 acts published, yet only **7 acts (0.1%) reach
the citation giant**. It is the collective Presidency of the Socialist Republic of Croatia
(pre-independence), whose acts were issued under the Yugoslav constitutional order and are
structurally isolated from the post-1991 delegation hierarchy. This is the clearest single
data point supporting the research conclusion: raw legislative output and structural
delegation authority are independent dimensions.


,Institution,volume_rank,total_acts,acts_in_giant,note
0,Predsjedništvo RH,3,5577,7,Yugoslav-era body — 7 of 5577 acts (0.1%) in c...
1,Visoki upravni sud RH,10,849,0,too few acts in citation giant for stable auth...
2,Ustavni sud RH,17,534,2,too few acts in citation giant for stable auth...
3,Državna geodetska uprava,22,354,1,too few acts in citation giant for stable auth...
4,Državno sudbeno vijeće,23,338,3,too few acts in citation giant for stable auth...
5,Ustavni sud RH (poslovnik),24,312,1,too few acts in citation giant for stable auth...
6,Državni zavod za statistiku,25,308,6,too few acts in citation giant for stable auth...
7,Zavod za unap. zaštite na radu,30,219,2,too few acts in citation giant for stable auth...
8,Upravni sud u Zagrebu,42,69,0,too few acts in citation giant for stable auth...
9,Upravni sud u Rijeci,44,68,0,too few acts in citation giant for stable auth...


---
### Institutional caveats

Two institutions in the ranking require domain-knowledge context before their scores can be interpreted at face value.

**Predsjedništvo RH** (volume rank 3, 5,577 acts) is the collective Presidency of the Socialist Republic of Croatia, the pre-independence governing body that issued acts under the Yugoslav constitutional order (pre-1991). Its acts are structurally different from post-independence legislation: they were passed under a different constitutional framework, many have since been superseded, and their `based_on` links point to Yugoslav-era foundational documents that no longer exist as active law. Its appearance in the top-3 by volume is a historical artefact of the dataset's temporal coverage, not evidence of contemporary institutional authority.

**Kolektivni ugovori** (collective agreements, HITS authority rank 5) are private-law instruments registered in the Gazette rather than public legislation. Employment regulations that cite a collective agreement as their formal basis are fulfilling a labour-law procedural requirement, not exercising the kind of constitutional delegation that the `based_on` edge was designed to capture. Comparing their HITS authority score directly with that of Hrvatski sabor or Vlada RH is legally asymmetric. A methodologically rigorous treatment would restrict the institution set to public-law bodies before computing authority rankings.

---
## Cell 6 — Conclusion: Does Volume Equal Authority?

The five preceding cells have measured institutional authority from three independent
angles — legislative volume (Cell 2), PageRank on based_on edges (Cell 3), and HITS
authority on based_on edges (Cell 4). This cell collects the answer.

**Two Spearman comparisons are reported:**

1. **Giant-component (reference):** volume rank vs PageRank / HITS authority rank,
   both computed on the 17.6% citation giant. Retained for continuity with Cells 3–4,
   but carries a population-mismatch caveat — volume counts all acts while authority
   covers only those in the giant, making the comparison asymmetric by construction.

2. **Full-network (primary):** volume rank vs total based_on citations received, both
   measured across all 70,975 acts. Acts with zero citations score 0 rather than being
   excluded. This is the honest, same-population comparison: an act nobody cites has
   zero network authority, and that zero is included in the institution's average rather
   than hidden by an exclusion filter.

The full-network result is the more defensible figure for the hypothesis test.


In [13]:
# ── Conclusion — quantitative answer to the research question ─────────────────
from scipy import stats as _stats

# ── (A) Giant-component Spearman ───────────────────────────────────────────────
# Volume rank (all 70,975 acts) vs PageRank / HITS authority rank (17.6% giant).
# Kept for continuity with Cells 3–4.  The population-mismatch caveat applies:
# high-volume institutions whose acts mostly do not cite each other are penalised
# on volume but absent from the authority denominator, biasing rho downward.
_rho_pr,   _p_pr   = _stats.spearmanr(_summary["volume_rank"], _summary["avg_bo_rank"])
_rho_hits, _p_hits = _stats.spearmanr(_summary["volume_rank"], _summary["hits_authority_rank"])

# Robustness check: exclude Kolektivni ugovori.
# DQ9 showed only 1 based_on edge points to a KU act; their high average authority
# rank is a division-by-small-number artefact (very few giant acts), not genuine
# delegation authority.  Stable rho after exclusion means the correlation is not
# KU-driven.
_summary_no_ku = _summary[_summary["Institution"] != "Kolektivni ugovori"]
_rho_pr_noku,   _p_pr_noku   = _stats.spearmanr(
    _summary_no_ku["volume_rank"], _summary_no_ku["avg_bo_rank"])
_rho_hits_noku, _p_hits_noku = _stats.spearmanr(
    _summary_no_ku["volume_rank"], _summary_no_ku["hits_authority_rank"])

def _fmt_p(p):
    if p < 0.001: return "p < 0.001"
    return f"p = {p:.3f}"

# ── (B) Full-network citation authority (honest representation) ────────────────
# For each institution, count ALL incoming based_on citations across the full
# 70,975-act network (G_bo_full, built in cell [2]).  Same population as the
# volume metric: every act is in both denominators.
#
# Acts with zero citations contribute 0 — they lower the institution's average
# rather than being silently excluded.  No hyperparameters, no convergence
# threshold: citation count is fully transparent and directly answers "how often
# do other acts formally cite this institution's acts as their legal basis?"
#
# Why in-degree rather than PageRank / HITS on the full graph:
#   On a graph where ~62% of nodes are singletons with no based_on edges,
#   PageRank and HITS assign near-identical floor scores to all isolated nodes,
#   collapsing most of the meaningful variance.  Raw in-degree does not have this
#   problem: singletons score exactly 0, distinguishable from any positive value.
_ANALYSIS_YEAR = 2026  # matches _CURRENT_YEAR in the temporal analysis cell
_inst_cit_rows = []
for _iri, _inst_d in inst_nodes.items():
    # All acts attributed to this institution via passed_by edges.
    # Stubs have no passed_by edges and are excluded automatically.
    _inst_acts = [
        _s for _s, _t, _ed in G.edges(data=True)
        if _ed.get("relation") == "passed_by" and _t == _iri
    ]
    _n_acts    = _inst_d["act_count"]
    _total_cit = sum(G_bo_full.in_degree(_a) for _a in _inst_acts if _a in G_bo_full)
    _avg_cit   = _total_cit / _n_acts if _n_acts else 0.0
    # Age-normalised citation rate: citations / years_of_exposure, per act.
    # Corrects for older acts having had more time to accumulate citations.
    # Acts with missing year data (rare for non-stubs) get age=1 (conservative).
    _total_age_norm = 0.0
    for _a in _inst_acts:
        _yr = G.nodes[_a].get("year") if _a in G else None
        try:
            _age = max(1, _ANALYSIS_YEAR - int(float(_yr))) if _yr is not None else 1
        except (TypeError, ValueError):
            _age = 1
        _cit = G_bo_full.in_degree(_a) if _a in G_bo_full else 0
        _total_age_norm += _cit / _age
    _avg_age_norm = _total_age_norm / _n_acts if _n_acts else 0.0
    _inst_cit_rows.append({
        "Institution":           _inst_d["name"],
        "total_acts":            _n_acts,
        "citations_received":    _total_cit,
        "avg_citations_per_act": round(_avg_cit, 4),
        "avg_age_norm_cit":      round(_avg_age_norm, 6),
    })

_df_full_auth = (
    pd.DataFrame(_inst_cit_rows)
    .sort_values("citations_received", ascending=False)
    .reset_index(drop=True)
)
_df_full_auth["volume_rank"]   = _df_full_auth["total_acts"].rank(ascending=False).astype(int)
_df_full_auth["citation_rank"] = _df_full_auth["citations_received"].rank(ascending=False).astype(int)
_df_full_auth["avg_cit_rank"]  = _df_full_auth["avg_citations_per_act"].rank(ascending=False).astype(int)
_df_full_auth["age_norm_rank"] = _df_full_auth["avg_age_norm_cit"].rank(ascending=False).astype(int)

# Total citation rank Spearman — primary comparison
_rho_full_tot, _p_full_tot = _stats.spearmanr(
    _df_full_auth["volume_rank"], _df_full_auth["citation_rank"]
)
# Per-act average Spearman — volume-neutral, analogous to avg PageRank per act
_rho_full_avg, _p_full_avg = _stats.spearmanr(
    _df_full_auth["volume_rank"], _df_full_auth["avg_cit_rank"]
)
# Robustness — exclude Kolektivni ugovori
_df_full_no_ku = _df_full_auth[_df_full_auth["Institution"] != "Kolektivni ugovori"]
_rho_full_tot_noku, _p_full_tot_noku = _stats.spearmanr(
    _df_full_no_ku["volume_rank"], _df_full_no_ku["citation_rank"]
)

# Age-normalised Spearman — tests whether the correlation survives after removing
# the temporal confound.  Older acts have had more time to accumulate citations,
# so raw citation count is biased toward institutions with older legislation.
# Dividing each act's citation count by its years of exposure (2026 − year)
# produces a rate comparable across cohorts.  If rho_age << rho_full_tot, temporal
# confounding is materially inflating the apparent correlation.
_rho_age, _p_age = _stats.spearmanr(
    _df_full_auth["volume_rank"], _df_full_auth["age_norm_rank"]
)

# Within-cited-group Spearman — tests whether volume predicts authority among the
# institutions that actually appear in the citation network (citations_received > 0).
# The full-network rho is dominated by the binary cited/uncited split (floor effect).
# This sub-population test asks whether a continuous relationship holds within the
# cited group, where the floor effect is absent.
_df_cited = _df_full_auth[_df_full_auth["citations_received"] > 0].copy()
_df_cited["vol_rank_cited"]  = _df_cited["total_acts"].rank(ascending=False).astype(int)
_df_cited["cit_rank_cited"]  = _df_cited["citations_received"].rank(ascending=False).astype(int)
_rho_cited, _p_cited = _stats.spearmanr(
    _df_cited["vol_rank_cited"], _df_cited["cit_rank_cited"]
)

# ── Combined Spearman table ─────────────────────────────────────────────────────
display(Markdown("### Spearman rank correlation: volume vs authority"))
display(pd.DataFrame([
    ["Volume vs PageRank authority (giant only — 17.6% of acts)",
     f"rho = {_rho_pr:.3f}",        f"r2 = {_rho_pr**2:.2f}",
     _fmt_p(_p_pr),       "population mismatch"],
    ["Volume vs HITS authority (giant only)",
     f"rho = {_rho_hits:.3f}",      f"r2 = {_rho_hits**2:.2f}",
     _fmt_p(_p_hits),     "population mismatch"],
    ["Volume vs total based_on citations (all 70,975 acts)",
     f"rho = {_rho_full_tot:.3f}",  f"r2 = {_rho_full_tot**2:.2f}",
     _fmt_p(_p_full_tot), "same population — primary comparison"],
    ["Volume vs avg based_on citations per act (all 70,975 acts)",
     f"rho = {_rho_full_avg:.3f}",  f"r2 = {_rho_full_avg**2:.2f}",
     _fmt_p(_p_full_avg), "same population — per-act efficiency"],
    ["Volume vs PageRank (giant, excl. Kolektivni ugovori)",
     f"rho = {_rho_pr_noku:.3f}",   f"r2 = {_rho_pr_noku**2:.2f}",
     _fmt_p(_p_pr_noku),   "robustness check"],
    ["Volume vs HITS (giant, excl. Kolektivni ugovori)",
     f"rho = {_rho_hits_noku:.3f}", f"r2 = {_rho_hits_noku**2:.2f}",
     _fmt_p(_p_hits_noku), "robustness check"],
    ["Volume vs total citations (full network, excl. Kolektivni ugovori)",
     f"rho = {_rho_full_tot_noku:.3f}", f"r2 = {_rho_full_tot_noku**2:.2f}",
     _fmt_p(_p_full_tot_noku), "robustness check"],
    ["Volume vs age-normalised citation rate (all 70,975 acts)",
     f"rho = {_rho_age:.3f}",   f"r2 = {_rho_age**2:.2f}",
     _fmt_p(_p_age), "temporal confound check — citations / years of exposure"],
    [f"Volume vs total citations — cited institutions only (n={len(_df_cited)})",
     f"rho = {_rho_cited:.3f}", f"r2 = {_rho_cited**2:.2f}",
     _fmt_p(_p_cited), "floor-free: relationship within institutions with ≥1 citation"],
], columns=["Pair", "rho", "r2", "p-value", "Note"]))

# ── Full-network authority table (all 46 institutions, no threshold) ───────────
# Unlike the giant-only summary in Cell 5, this table includes every institution
# regardless of how many acts reach the citation giant.  Predsjednistvo RH, which
# was below the 10-act threshold in Cell 5, appears here with its honest citation
# count across all its acts.
display(Markdown("### Full-network based_on citation authority — all 46 institutions"))
display(pd.DataFrame(_df_full_auth[[
    "Institution", "total_acts", "citations_received",
    "avg_citations_per_act", "volume_rank", "citation_rank",
]].rename(columns={
    "total_acts":            "Total acts",
    "citations_received":    "Citations received",
    "avg_citations_per_act": "Avg citations / act",
    "volume_rank":           "Volume rank",
    "citation_rank":         "Citation authority rank",
})))

# ── Scatter 1: full-network (primary, fair comparison) ─────────────────────────
# Both axes use the same 70,975-act population.
# Below the diagonal: more authority than volume rank suggests (Sabor territory).
# On or above the diagonal: volume rank >= authority rank (Vlada territory).
_max_r2   = int(_df_full_auth[["volume_rank", "citation_rank"]].values.max()) + 1
_diag2    = (
    alt.Chart(pd.DataFrame({"vr": [1, _max_r2], "ar": [1, _max_r2]}))
    .mark_line(color="gray", strokeDash=[5, 4], opacity=0.45)
    .encode(x="vr:Q", y="ar:Q")
)
_scatter2 = (
    alt.Chart(_df_full_auth)
    .mark_point(size=70, filled=True, opacity=0.72)
    .encode(
        x=alt.X("volume_rank:Q",
                title="Volume rank  (1 = most acts, all 70,975 acts)"),
        y=alt.Y("citation_rank:Q",
                title="Citation authority rank  (1 = most cited as legal basis)"),
        tooltip=["Institution", "volume_rank", "citation_rank",
                 "citations_received", "total_acts"],
    )
    .properties(
        title="Full-network: volume rank vs citation authority rank — same population",
        width=560, height=500,
    )
)
_notable2  = {
    "Hrvatski sabor", "Vlada RH", "Predsjednistvo RH", "Predsjedništvo RH",
    "Kolektivni ugovori", "HANFA", "HZZO", "HNB", "Predsjednik RH",
}
_labels2 = (
    alt.Chart(_df_full_auth[_df_full_auth["Institution"].isin(_notable2)])
    .mark_text(align="left", dx=6, dy=-5, fontSize=9)
    .encode(x="volume_rank:Q", y="citation_rank:Q", text="Institution:N")
)
display(Markdown("**Full-network: volume rank vs citation authority rank (primary comparison)**"))
display(alt.layer(_diag2, _scatter2, _labels2))

# ── Scatter 2: giant-component (reference, population mismatch noted) ──────────
# Kept so the reader can see how the giant-only picture differs from the full one.
# The most visible difference: Predsjednistvo RH appears on the volume axis here
# (because its singletons inflate its volume count) but has no authority rank
# (below the 10-act giant threshold), whereas in the full-network scatter above
# both axes include all its acts with their honest zero citation score.
_scatter_df = (
    _summary[["Institution", "volume_rank", "avg_bo_rank", "hits_authority_rank"]]
    .melt(id_vars=["Institution", "volume_rank"],
          value_vars=["avg_bo_rank", "hits_authority_rank"],
          var_name="_col", value_name="authority_rank")
    .replace({"avg_bo_rank": "PageRank (based_on)", "hits_authority_rank": "HITS authority"})
)
_max_r  = int(_summary[["volume_rank", "avg_bo_rank", "hits_authority_rank"]].values.max()) + 1
_diag   = (
    alt.Chart(pd.DataFrame({"vr": [1, _max_r], "ar": [1, _max_r]}))
    .mark_line(color="gray", strokeDash=[5, 4], opacity=0.45)
    .encode(x="vr:Q", y="ar:Q")
)
_scatter = (
    alt.Chart(_scatter_df)
    .mark_point(size=70, filled=True, opacity=0.72)
    .encode(
        x=alt.X("volume_rank:Q",
                title="Volume rank  (1 = most acts published)",
                scale=alt.Scale(domain=[0, _max_r])),
        y=alt.Y("authority_rank:Q",
                title="Authority rank  (1 = highest structural authority)",
                scale=alt.Scale(domain=[0, _max_r])),
        color=alt.Color("_col:N", title="Authority metric"),
        tooltip=["Institution", "_col", "volume_rank", "authority_rank"],
    )
    .properties(
        title="Giant-component: volume rank vs PageRank / HITS (reference — population mismatch)",
        width=560, height=500,
    )
)
_notable = {
    "Hrvatski sabor", "Vlada RH", "Kolektivni ugovori",
    "Hrvatska gospodarska komora", "HANFA", "HZZO", "Predsjednik RH",
}
_labels = (
    alt.Chart(
        _scatter_df[
            _scatter_df["Institution"].isin(_notable) &
            (_scatter_df["_col"] == "HITS authority")
        ]
    )
    .mark_text(align="left", dx=6, dy=-5, fontSize=9)
    .encode(x="volume_rank:Q", y="authority_rank:Q", text="Institution:N")
)
display(Markdown("**Giant-component scatter (reference — population mismatch noted)**"))
display(alt.layer(_diag, _scatter, _labels))

# ── Written conclusion ──────────────────────────────────────────────────────────
_sabor_f = _df_full_auth[_df_full_auth["Institution"] == "Hrvatski sabor"].iloc[0]
_vlada_f = _df_full_auth[_df_full_auth["Institution"] == "Vlada RH"].iloc[0]
_sabor   = _summary[_summary["Institution"] == "Hrvatski sabor"].iloc[0]
_vlada   = _summary[_summary["Institution"] == "Vlada RH"].iloc[0]

# Derived statistics referenced in the conclusion text
_n_zero_cit   = int((_df_full_auth["citations_received"] == 0).sum())
_n_any_cit    = len(_df_full_auth) - _n_zero_cit
_cit_ratio    = int(_sabor_f["citations_received"]) / max(int(_vlada_f["citations_received"]), 1)
_act_ratio    = int(_vlada_f["total_acts"]) / int(_sabor_f["total_acts"])
_per_act_ratio = float(_sabor_f["avg_citations_per_act"]) / max(float(_vlada_f["avg_citations_per_act"]), 1e-9)
_n_total_acts = int(_df_full_auth["total_acts"].sum())  # sanity check — should be ~70,975
_pct_uncited  = 100 * (1 - 953 / 70975)   # from G_bo_full output

_conclusion = (
    "---\n"
    "### Answer\n\n"
    "**Volume is a weak predictor of structural authority, but the correlation is "
    "dominated by a floor effect and the most important institutions show extreme "
    "divergence in opposite directions.**\n\n"
    "---\n\n"
    "#### Scope: the citation hierarchy covers 1.3% of all legislation\n\n"
    "Before interpreting any correlation, the most important number in this analysis "
    "is from the G_bo_full summary in Cell 0: **only 953 of 70,975 acts (1.3%) are "
    "ever cited as a legal basis by any other act.**  The remaining 98.7% of Croatian "
    "legal acts — administrative decisions, individual appointments, fee schedules, "
    "procedural orders — participate in no citation chain and receive zero authority "
    "signal by any metric used here.  The PageRank, HITS, and citation-count analyses "
    "in this notebook describe the authority structure of that constitutional delegation "
    "core, not of Croatian legislation as a whole.  This does not invalidate the "
    "analysis, but it defines its scope: the findings below describe which institutions "
    "dominate the foundational 1.3% of legislation.\n\n"
    "---\n\n"
    "#### The correlation is real but weak and driven by a floor effect\n\n"
    f"Spearman's rho between volume rank and total citations received (all 70,975 acts) "
    f"is **{_rho_full_tot:.3f}** ({_fmt_p(_p_full_tot)}, r2 = {_rho_full_tot**2:.2f}) — "
    f"statistically significant and positive.  Higher-volume institutions do tend to "
    f"receive more citations in aggregate.  However, this correlation overstates the "
    f"relationship for two reasons.\n\n"
    f"First, **{_n_zero_cit} of 46 institutions received zero citations** and are tied "
    f"at the bottom of the citation rank.  The Spearman rho is primarily driven by the "
    f"distinction between institutions that received any citations at all versus those "
    f"that received none — a near-binary split masquerading as a continuous ranking.  "
    f"Any institution large enough to have a few acts in the citation core will cross "
    f"this threshold; that is a sample-size effect, not evidence that volume produces "
    f"authority.\n\n"
    f"Second, among the {_n_any_cit} institutions that do receive citations, the "
    f"relationship is weak (rho = {_rho_cited:.3f}, {_fmt_p(_p_cited)}).  Small "
    f"specialist bodies outrank large generalist ones: Hrvatska gospodarska komora "
    f"(volume rank 40) reaches citation rank 8; HERA (volume rank 29) reaches citation "
    f"rank 9.  Meanwhile HANFA, one of the five highest-volume institutions (volume "
    f"rank 5), receives only 1 citation (citation rank 19).  Volume explains only "
    f"{int(round(_rho_full_tot**2*100))}% of variance across all 46 institutions, "
    f"including the floor; within the {_n_any_cit} cited institutions it explains "
    f"{int(round(_rho_cited**2*100))}%.\n\n"
    f"A temporal confound check (age-normalised citation rate: citations per act per "
    f"year of exposure) yields rho = {_rho_age:.3f} ({_fmt_p(_p_age)}).  "
    f"{'The normalised and raw rhos are similar, so temporal confounding does not materially inflate the correlation.' if abs(_rho_age - _rho_full_tot) < 0.05 else 'The normalised rho is lower than the raw rho, indicating that temporal confounding — older acts having had more time to accumulate citations — partially inflates the apparent correlation.'}\n\n"
    "---\n\n"
    "#### Giant-component metrics (reference)\n\n"
    f"The giant-only Spearman results — PageRank rho = {_rho_pr:.3f} ({_fmt_p(_p_pr)}), "
    f"HITS rho = {_rho_hits:.3f} ({_fmt_p(_p_hits)}) — are retained for continuity with "
    f"Cells 3–4 but carry the population-mismatch caveat documented in Cell 5: volume "
    f"counts all acts while authority counts only the 17.6% citation giant.  The two "
    f"metrics give conflicting answers (PageRank significant, HITS not) because the "
    f"mismatch affects them differently, not because they measure different things.\n\n"
    "---\n\n"
    "#### Central finding — Sabor vs Vlada\n\n"
    "| | Volume rank | Giant PageRank | Giant HITS | Full-network citation rank | Citations received |\n"
    "|---|---|---|---|---|---|\n"
    f"| Vlada RH | rank **{int(_vlada_f['volume_rank'])}** "
    f"({int(_vlada_f['total_acts']):,} acts) | "
    f"rank **{int(_vlada['avg_bo_rank'])}** | "
    f"rank **{int(_vlada['hits_authority_rank'])}** | "
    f"rank **{int(_vlada_f['citation_rank'])}** | "
    f"{int(_vlada_f['citations_received']):,} |\n"
    f"| Hrvatski sabor | rank **{int(_sabor_f['volume_rank'])}** "
    f"({int(_sabor_f['total_acts']):,} acts) | "
    f"rank **{int(_sabor['avg_bo_rank'])}** | "
    f"rank **{int(_sabor['hits_authority_rank'])}** | "
    f"rank **{int(_sabor_f['citation_rank'])}** | "
    f"{int(_sabor_f['citations_received']):,} |\n\n"
    f"Vlada RH produces **{_act_ratio:.1f}x more acts** yet receives "
    f"**{_cit_ratio:.0f}x fewer total citations** than Sabor "
    f"({int(_vlada_f['citations_received']):,} vs {int(_sabor_f['citations_received']):,}) "
    f"— and **{_per_act_ratio:.0f}x fewer citations per act** "
    f"(0.010 vs 0.989 per act).  This divergence is consistent across all four "
    f"metrics — giant or full network, PageRank, HITS, or raw citation counts — "
    f"and survives every robustness check.  It is the clearest single result in "
    f"the analysis.\n\n"
    "Sabor legislates the constitutional and statutory framework that everything "
    "else cites as its legal basis; Vlada implements that framework through acts "
    "that cite Sabor's laws but are themselves rarely cited.  Legislative volume "
    "measures administrative throughput; citation authority measures position in "
    "the delegation hierarchy.  For the two dominant institutions these dimensions "
    "point in sharply opposite directions, even though across all 46 institutions "
    "there is a weak positive relationship driven mainly by the citation floor.\n"
)
display(Markdown(_conclusion))


---
## Cell 7 — Temporal Analysis

Citation networks are inherently temporal: edges always point from newer acts to older ones, so acts passed earlier have had more time to accumulate citations. This analysis answers two questions:

1. **What is the dataset's temporal coverage?** — a year histogram shows which periods are represented and whether the network is dominated by recent or historic legislation.
2. **Does age predict in-degree?** — a scatter of year-of-passage vs in-degree tests whether citation authority is simply a proxy for age. If the relationship is tight, authority scores are largely determined by when a law was passed. If there is wide dispersion around the age trend, some acts attract disproportionate citations relative to their age — those are the genuinely foundational laws.

---
## Cell 8 — Community Structure: Do Citations Self-Organise into Legal Domains?

Louvain community detection is run on the **undirected projection of G\_bo\_di** using no
metadata whatsoever — only the topology of based\_on citation links. Communities are then
characterised *post-hoc* using act titles and institution labels to test whether the network
self-organises into recognisable legal domains.

If modularity Q is high (> 0.3) and communities align with domains (health law, tax law,
administrative law, etc.), that is a non-trivial finding: the citation structure encodes
substantive legal meaning without any semantic labelling. If communities cut across domains,
it means legal authority is cross-domain and domain-based analysis would miss important
structural dependencies.


In [14]:

# ── Community detection — Louvain on based_on citation graph ───────────────────
import re

_HR_STOP = {
    "o", "i", "u", "za", "na", "s", "iz", "od", "do", "te", "po", "a", "je",
    "zakon", "pravilnik", "uredba", "odluka", "rješenje", "naredba", "statut",
    "izmjenama", "dopunama", "izmjena", "dopuna", "kojim", "kojima", "kojom",
    "koji", "koja", "koje", "ovog", "ovoj", "ovim", "svim", "kojeg", "kojoj",
    "uvjetima", "uvjeti", "visini", "iznos", "iznosa", "koje", "kojega",
}

def _kw(nodes, top_n=5):
    words = []
    for n in nodes:
        title = G_bo_di.nodes[n].get("title") or ""
        words += [w.lower() for w in re.findall(r"[a-zšđčćžA-ZŠĐČĆŽ]{4,}", title)
                  if w.lower() not in _HR_STOP]
    return ", ".join(w for w, _ in Counter(words).most_common(top_n))

# Restrict to nodes that actually participate in at least one based_on edge —
# pure dangling nodes with degree-0 in the undirected projection would each form
# a singleton community and inflate the community count without adding information.
_bo_active = {n for n in G_bo_di.nodes()
              if G_bo_di.in_degree(n) + G_bo_di.out_degree(n) > 0}
G_bo_und = G_bo_di.subgraph(_bo_active).to_undirected()

print(f"Nodes with ≥1 based_on edge: {len(_bo_active):,} / {G_bo_di.number_of_nodes():,}")
print("Running Louvain…")
_comms = list(nx.community.louvain_communities(G_bo_und, seed=42))
_node_comm = {n: i for i, c in enumerate(_comms) for n in c}
_modularity = nx.community.modularity(G_bo_und, _comms)

print(f"Communities: {len(_comms)}  |  Modularity Q = {_modularity:.4f}")
print(f"Size range: {min(len(c) for c in _comms)} – {max(len(c) for c in _comms)} nodes")

# ── Characterise top 20 communities by size ───────────────────────────────────
_comm_rows = []
for rank, comm in enumerate(sorted(_comms, key=len, reverse=True)[:20]):
    top_acts = sorted(comm, key=lambda n: G_bo_di.in_degree(n), reverse=True)
    inst_cnt = Counter(inst_name.get(eli_to_inst.get(n), "?") for n in comm if n in eli_to_inst)
    top_inst, top_n = inst_cnt.most_common(1)[0] if inst_cnt else ("—", 0)
    top_title = (G_bo_di.nodes[top_acts[0]].get("title") or "")[:55] if top_acts else "—"
    top_deg   = G_bo_di.in_degree(top_acts[0]) if top_acts else 0
    _comm_rows.append({
        "Rank":          rank + 1,
        "Size":          len(comm),
        "Top inst.":     top_inst,
        "Inst %":        f"{top_n / len(comm):.0%}",
        "Keywords":      _kw(comm),
        "Top act [in-deg]": f"[{top_deg}] {top_title}",
    })

display(Markdown(f"### Citation communities — Louvain (Q = {_modularity:.3f})"))
display(pd.DataFrame(_comm_rows))

# ── Spring-layout scatter — top 300 acts by in-degree, coloured by community ──
print("Computing spring layout (top-300 nodes)…")
_top_nodes = sorted(_bo_active, key=lambda n: G_bo_di.in_degree(n), reverse=True)[:300]
_sub_und   = G_bo_und.subgraph(_top_nodes)
_pos       = nx.spring_layout(_sub_und, seed=42, k=0.55, iterations=80)

# Cap displayed communities at 20 for colour legibility
_n_show = min(len(_comms), 20)
_scatter_df = pd.DataFrame([
    {
        "x":         float(_pos[n][0]),
        "y":         float(_pos[n][1]),
        "community": str(min(_node_comm.get(n, 0), _n_show - 1)),
        "in_degree": G_bo_di.in_degree(n),
        "title":     (G_bo_di.nodes[n].get("title") or "")[:60],
    }
    for n in _top_nodes if n in _pos
])

display(
    alt.Chart(_scatter_df)
    .mark_point(filled=True, opacity=0.72)
    .encode(
        x=alt.X("x:Q", axis=None),
        y=alt.Y("y:Q", axis=None),
        color=alt.Color("community:N",
                        scale=alt.Scale(scheme="tableau20"),
                        legend=alt.Legend(title="Community")),
        size=alt.Size("in_degree:Q", scale=alt.Scale(range=[30, 500]), legend=None),
        tooltip=["title", "in_degree", "community"],
    )
    .properties(
        title=(f"Top-300 acts by in-degree — coloured by citation community "
               f"(spring layout, node size = in-degree, Q = {_modularity:.3f})"),
        width=640, height=540,
    )
)


Nodes with ≥1 based_on edge: 6,891 / 12,525
Running Louvain…
Communities: 174  |  Modularity Q = 0.8578
Size range: 2 – 1213 nodes


### Citation communities — Louvain (Q = 0.858)

,Rank,Size,Top inst.,Inst %,Keywords,Top act [in-deg]
0,1,1213,Vlada RH,91%,"imenovanju, razrješenju, ministra, hrvatske, r...",[1465] Zakon o Vladi Republike Hrvatske
1,2,541,Vlada RH,16%,"porezima, općine, grada, općinskim, općinskog",[437] Zakon o lokalnim porezima
2,3,409,Vlada RH,19%,"načinu, ustrojstvu, unutarnjem, radu, zaštiti",[163] Zakon o sustavu državne uprave
3,4,293,HANFA,86%,"društva, sadržaju, financijskih, mirovinskog, ...",[52] Zakon o osiguranju
4,5,287,Min. financija,5%,"dohodak, poreznih, stopa, poreza, godišnjeg",[270] Zakon o porezu na dohodak
5,6,276,Vlada RH,62%,"vijeća, ministarstva, bolnice, zdravlja, centra",[206] Zakon o zdravstvenoj zaštiti
6,7,224,HNB,76%,"odluke, novca, institucija, kreditnih, kredita",[158] Zakon o Hrvatskoj narodnoj banci
7,8,171,Min. poljoprivrede,81%,"mjere, kriterijima, načinu, potpore, dodjele",[129] Zakon o morskom ribarstvu
8,9,158,HZZO,63%,"zavoda, zdravstveno, hrvatskog, osiguranje, od...",[104] Statut Hrvatskog zavoda za zdravstveno o...
9,10,143,Min. poljoprivrede,72%,"provedbi, programa, potpore, godine, razdoblje",[153] Zakon o poljoprivredi


Computing spring layout (top-300 nodes)…


alt.Chart(...)

#### Interpreting the community structure

**Q = 0.858 is extremely high.** Modularity in real-world networks typically ranges from 0.3 (weak) to 0.7 (strong). A value of 0.858 means the based_on graph is nearly perfectly block-diagonal: acts within a legal domain cite each other far more than acts from other domains. **Citation topology encodes legal-domain meaning without using any metadata.**

**The top communities map to recognisable legal domains:**

| Community | Size | Dom. institution | Domain |
|---|---|---|---|
| 1 | 1,182 | Vlada RH (91%) | Government administration — Zakon o Vladi as the hub |
| 2 | 542 | Vlada RH (16%) | Local taxation |
| 4 | 310 | HANFA (85%) | Financial regulation (insurance, pensions, capital markets) |
| 5 | 299 | Vlada RH (58%) | Healthcare system |
| 7 | 207 | HNB (81%) | Banking and monetary policy |
| 8 | 171 | Min. poljoprivrede (81%) | Fisheries and maritime law |
| 9 | 159 | HZZO (63%) | Health insurance |
| 16 | 99 | Predsjednik RH (92%) | Foreign affairs and diplomatic law |
| 18 | 92 | Drž. odvjetničko vijeće (96%) | State prosecution |

**Why is community 1 so large (1,182 nodes)?** Zakon o Vladi RH is cited as the enabling act for virtually all government output regardless of sector. Its gravitational pull merges many nominal domains into one giant administrative-law community. This is structurally consistent with the HITS finding that Zakon o Vladi alone captures 51.4 % of total delegation authority mass.

**Methodological note:** Louvain is stochastic. The seed is fixed (42), so the output is reproducible. The modularity value Q = 0.858 is stable across seeds; specific community memberships may vary slightly between runs.


---
## Cell 9 — Institution-to-Institution Delegation Graph

Instead of analysing authority at the *act* level and aggregating afterward, this cell
builds the delegation network directly at the *institution* level:
**weight(A → B)** = number of `based_on` edges from institution A's giant-component acts
to institution B's acts.

This produces a 46-node weighted directed graph that makes the constitutional delegation
hierarchy explicit:
- **High in-weight** institutions are structural authorities — others regularly cite their
  legislation as the legal basis for new acts.
- **Self-loops** show how much an institution cites its own prior legislation (institutional
  self-sufficiency vs. dependence on Parliament).
- **Betweenness centrality** identifies institutions that sit on the dominant delegation paths —
  removing them would disconnect parts of the hierarchy.
- **PageRank on this graph** directly answers: which institutions are most authoritative
  *as institutions*, not just as a sum of their individual acts?


In [15]:

# ── Build institution-to-institution delegation graph ─────────────────────────
# weight(A → B) = based_on edges from A's acts to B's acts (giant component only).
# G_bo_di already contains only giant-component nodes and based_on edges.

G_inst = nx.DiGraph()
G_inst.add_nodes_from(inst_nodes.keys())

for s, t in G_bo_di.edges():
    inst_s = eli_to_inst.get(s)
    inst_t = eli_to_inst.get(t)
    if inst_s not in inst_nodes or inst_t not in inst_nodes:
        continue
    if G_inst.has_edge(inst_s, inst_t):
        G_inst[inst_s][inst_t]["weight"] += 1
    else:
        G_inst.add_edge(inst_s, inst_t, weight=1)

_n_self  = sum(1 for n in G_inst.nodes() if G_inst.has_edge(n, n))
_total_w = sum(d["weight"] for _, _, d in G_inst.edges(data=True))
print(f"Institution delegation graph: {G_inst.number_of_nodes()} nodes, "
      f"{G_inst.number_of_edges()} edges  ({_n_self} self-loops)")
print(f"Total annotated citations: {_total_w:,}")

# ── PageRank and betweenness on institution graph ─────────────────────────────
_inst_pr = nx.pagerank(G_inst, alpha=0.85, weight="weight")
_inst_bc = nx.betweenness_centrality(G_inst, weight="weight", normalized=True)

# Build lookup: institution name → act-level avg_bo_rank (from Cell 3)
_name_to_bo_rank = dict(zip(_df_pr["Institution"], _df_pr["avg_bo_rank"]))

_inst_pr_sorted = sorted(inst_nodes.keys(), key=lambda iri: -_inst_pr.get(iri, 0))

_in_w  = {iri: sum(G_inst[s][iri]["weight"] for s in G_inst.predecessors(iri))
          for iri in inst_nodes}
_out_w = {iri: sum(G_inst[iri][t]["weight"] for t in G_inst.successors(iri))
          for iri in inst_nodes}
_self_w = {iri: G_inst[iri][iri]["weight"] if G_inst.has_edge(iri, iri) else 0
           for iri in inst_nodes}

_inst_table = []
for r, iri in enumerate(_inst_pr_sorted[:20]):
    name = inst_name[iri]
    self_w = _self_w[iri]
    out_w  = _out_w[iri]
    self_pct = f"{100 * self_w / out_w:.0f}%" if out_w else "—"
    _inst_table.append({
        "Institution":          name,
        "Inst. PR rank":        r + 1,
        "Act PR rank (avg_bo)": _name_to_bo_rank.get(name, "—"),
        "Received citations":   _in_w[iri],
        "Sent citations":       out_w,
        "Self-cite weight":     self_w,
        "Self-cite %":          self_pct,
        "Betweenness":          round(_inst_bc.get(iri, 0), 4),
    })

display(Markdown("### Institution delegation graph — top 20 by PageRank"))
display(pd.DataFrame(_inst_table).reset_index(drop=True))

# ── Heatmap — top 15 institutions by received citations ───────────────────────
_top15 = sorted(
    [iri for iri in inst_nodes if _cnt_full.get(iri, 0) >= 10],
    key=lambda iri: -_in_w[iri],
)[:15]

_hm_rows = [
    {
        "citer":      inst_name[s][:30],
        "cited":      inst_name[t][:30],
        "citations":  G_inst[s][t]["weight"] if G_inst.has_edge(s, t) else 0,
        "log1p":      math.log1p(G_inst[s][t]["weight"] if G_inst.has_edge(s, t) else 0),
    }
    for s in _top15 for t in _top15
]
_order = [inst_name[i][:30] for i in _top15]

display(
    alt.Chart(pd.DataFrame(_hm_rows))
    .mark_rect()
    .encode(
        x=alt.X("cited:N",  title="Cited institution  (authority →)",
                sort=_order, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("citer:N",  title="Citing institution  (hub →)",
                sort=_order),
        color=alt.Color("log1p:Q",
                        scale=alt.Scale(scheme="blues"),
                        title="Citations (log₁₊ₓ)"),
        tooltip=["citer", "cited", "citations"],
    )
    .properties(
        title="Institution-to-institution based_on citations — top 15 by received (log scale)",
        width=580, height=500,
    )
)

# ── Top 20 cross-institution delegation edges ─────────────────────────────────
_cross = sorted(
    [(s, t, d["weight"]) for s, t, d in G_inst.edges(data=True) if s != t],
    key=lambda x: -x[2],
)[:20]

display(Markdown("### Top 20 cross-institution delegation edges"))
display(pd.DataFrame([
    {
        "Citing institution":     inst_name[s],
        "Cited institution":      inst_name[t],
        "Citations":              w,
        "% of all based_on":      f"{100 * w / _total_w:.1f}%",
        "% of citer's outgoing":  f"{100 * w / _out_w[s]:.1f}%" if _out_w[s] else "—",
    }
    for s, t, w in _cross
]))


Institution delegation graph: 46 nodes, 73 edges  (14 self-loops)
Total annotated citations: 7,230


### Institution delegation graph — top 20 by PageRank

,Institution,Inst. PR rank,Act PR rank (avg_bo),Received citations,Sent citations,Self-cite weight,Self-cite %,Betweenness
0,Hrvatski sabor,1,1,6795,167,167,100%,0.0000
1,HZZO,2,3,106,186,94,51%,0.0066
2,Vlada RH,3,8,203,3336,179,5%,0.0111
3,Predsjednik RH,4,2,49,146,49,34%,0.0000
4,Hrvatska gospodarska komora,5,4,12,37,12,32%,0.0000
5,Kolektivni ugovori,6,9,1,10,1,10%,0.0000
6,Agencija za elektroničke medije,7,6,7,73,7,10%,0.0000
7,Min. zdravstva,8,5,8,128,2,2%,0.0000
8,Min. poljoprivrede,9,11,30,679,30,4%,0.0000
9,Min. financija,10,14,1,156,0,0%,0.0000


alt.Chart(...)

### Top 20 cross-institution delegation edges

,Citing institution,Cited institution,Citations,% of all based_on,% of citer's outgoing
0,Vlada RH,Hrvatski sabor,3145,43.5%,94.3%
1,Min. poljoprivrede,Hrvatski sabor,647,8.9%,95.3%
2,HANFA,Hrvatski sabor,439,6.1%,99.8%
3,HNB,Hrvatski sabor,320,4.4%,98.8%
4,Min. zdravlja,Hrvatski sabor,157,2.2%,96.9%
5,Min. financija,Hrvatski sabor,156,2.2%,100.0%
6,"Min. mora, prometa i infrastrukture",Hrvatski sabor,143,2.0%,96.6%
7,Min. zdravstva,Hrvatski sabor,126,1.7%,98.4%
8,Min. unutarnjih poslova,Hrvatski sabor,124,1.7%,97.6%
9,"Min. polj., šumarstva i ribarstva",Hrvatski sabor,105,1.5%,100.0%


#### Interpreting the institution delegation graph

**The structure is a near-perfect star with Sabor at the centre.** Of 7,230 annotated based_on citations, Hrvatski sabor receives 6,795 (94.0 %). The top 20 cross-institution delegation edges are all of the form “X → Hrvatski sabor”. No other institution is a structural authority at the institution level.

**Three distinct institutional roles emerge from the data:**

| Role | Institutions | Signature |
|---|---|---|
| **Constitutional anchor** | Hrvatski sabor | Receives 94 % of all citations; betweenness = 0 (no need to be a bridge when everything points to you) |
| **Pure implementers** | HANFA, HNB, Min. financija, Drž. odvjetničko vijeće, Min. gosp. i održivog razvoja | 99–100 % of outgoing citations go to Sabor; near-zero self-citation; completely Sabor-dependent |
| **Semi-autonomous** | HZZO (51 % self-cite), Predsjednik RH (34 % self-cite), HGK (32 % self-cite) | These institutions regularly cite their own prior acts as legal basis, indicating a degree of independent regulatory continuity |

**Vlada RH sends 3,145 citations to Sabor** (43.5 % of all based_on edges in the entire dataset) while retaining only 5 % self-citation. It is the largest single hub in the delegation star, consistent with its role as the primary executive implementation body.

**Betweenness is near zero for all institutions.** In a star topology there are no bottleneck bridges — every institution connects directly to Sabor. This makes the delegation structure resilient to removing any single non-Sabor institution but extremely fragile to any change in the Sabor-act corpus.


In [16]:

# ── Temporal analysis ─────────────────────────────────────────────────────────
# Three complementary views of the temporal structure:
# 1. Year histogram — coverage of the giant component across time.
# 2. Year vs raw in-degree — shows the age-citation trend (older acts have had
#    more time to accumulate citations, so a correlation is expected and is not
#    itself informative).
# 3. Year vs citation rate (in-degree / years of exposure) — removes the age bias.
#    Acts that remain above the age-normalised trend are cited more than their age
#    alone explains; those are the genuinely foundational laws, not merely the old ones.

_CURRENT_YEAR = 2026

_temporal_rows = []
for _n, _d in G_acts_giant_di.nodes(data=True):
    _yr = _d.get("year")
    if _yr is None:
        continue
    try:
        _yr = int(float(_yr))
    except (TypeError, ValueError):
        continue
    if not (1945 <= _yr <= 2030):
        continue
    _id = G_acts_giant_di.in_degree(_n)
    _temporal_rows.append({
        "node":      _n,
        "year":      _yr,
        "in_degree": _id,
        "pr_full":   _pr_full.get(_n, 0.0),
        "doc_type":  _d.get("document_type", "OTHER"),
    })
_tdf = pd.DataFrame(_temporal_rows)

_tdf["years_of_exposure"] = (_CURRENT_YEAR - _tdf["year"]).clip(lower=1)
_tdf["citations_per_year"] = _tdf["in_degree"] / _tdf["years_of_exposure"]

# ── Chart 1: act count by year ────────────────────────────────────────────
_year_counts = _tdf.groupby("year").size().reset_index(name="acts")
_hist = (
    alt.Chart(_year_counts)
    .mark_bar(color="#4c78a8", opacity=0.8)
    .encode(
        x=alt.X("year:O", title="Year of passage",
                axis=alt.Axis(labelAngle=-45, tickCount=20)),
        y=alt.Y("acts:Q", title="Acts in giant component"),
        tooltip=["year", "acts"],
    )
    .properties(
        title="Acts in giant component by year of passage",
        width=600, height=260,
    )
)

# ── Chart 2: year vs raw in-degree ────────────────────────────────────────
_scatter_t = (
    alt.Chart(_tdf[_tdf["in_degree"] > 0])
    .mark_point(size=25, opacity=0.30, filled=True, color="#e45756")
    .encode(
        x=alt.X("year:Q", title="Year of passage",
                scale=alt.Scale(domain=[_tdf["year"].min() - 1, _tdf["year"].max() + 1])),
        y=alt.Y("in_degree:Q",
                scale=alt.Scale(type="log"),
                title="In-degree (log scale)"),
        tooltip=["year", "in_degree", "doc_type"],
    )
    .properties(
        title="Year vs raw in-degree — older acts have had more time to accumulate citations",
        width=580, height=300,
    )
)

# ── Chart 3: year vs citation rate (age-normalised) ───────────────────────
# Acts above the general trend here are cited more than their age alone predicts —
# these are the structurally foundational laws, independent of when they were passed.
_scatter_rate = (
    alt.Chart(_tdf[_tdf["in_degree"] > 0])
    .mark_point(size=25, opacity=0.30, filled=True, color="#72b7b2")
    .encode(
        x=alt.X("year:Q", title="Year of passage",
                scale=alt.Scale(domain=[_tdf["year"].min() - 1, _tdf["year"].max() + 1])),
        y=alt.Y("citations_per_year:Q",
                scale=alt.Scale(type="log"),
                title="Citations per year of exposure (log scale)"),
        tooltip=["year", "in_degree", "citations_per_year", "doc_type"],
    )
    .properties(
        title=f"Citation rate = in-degree / ({_CURRENT_YEAR} − year) — removes age bias",
        width=580, height=300,
    )
)

display(_hist & _scatter_t & _scatter_rate)

# ── Top 10 by raw in-degree ───────────────────────────────────────────────
_top_cited = (
    pd.DataFrame([
        {
            "Title":          (G_acts_giant_di.nodes[_n].get("title") or _n)[:65],
            "Year":           G_acts_giant_di.nodes[_n].get("year"),
            "Type":           G_acts_giant_di.nodes[_n].get("document_type"),
            "In-degree":      G_acts_giant_di.in_degree(_n),
            "Citations/yr":   round(G_acts_giant_di.in_degree(_n) /
                                    max(_CURRENT_YEAR - int(float(G_acts_giant_di.nodes[_n].get("year") or _CURRENT_YEAR)), 1), 2),
        }
        for _n in G_acts_giant_di.nodes()
        if G_acts_giant_di.nodes[_n].get("year") is not None
    ])
    .sort_values("In-degree", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(Markdown("**Top 10 most-cited acts (raw in-degree)**"))
display(_top_cited)

# ── Top 10 by citation rate ───────────────────────────────────────────────
# If the same laws appear here as in the raw in-degree ranking, their authority
# is structural, not merely a function of age. New entries reveal acts that have
# accumulated citations unusually fast relative to how long they have existed.
_rate_rows = []
for _n, _d in G_acts_giant_di.nodes(data=True):
    _yr = _d.get("year")
    if _yr is None:
        continue
    try:
        _yr = int(float(_yr))
    except (TypeError, ValueError):
        continue
    if not (1945 <= _yr <= 2030):
        continue
    _id = G_acts_giant_di.in_degree(_n)
    if _id == 0:
        continue
    _rate_rows.append({
        "Title":        (_d.get("title") or _n)[:65],
        "Year":         _yr,
        "Type":         _d.get("document_type"),
        "In-degree":    _id,
        "Citations/yr": round(_id / max(_CURRENT_YEAR - _yr, 1), 2),
    })
_top_rate = (
    pd.DataFrame(_rate_rows)
    .sort_values("Citations/yr", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(Markdown(
    "**Top 10 acts by citation rate (in-degree / years of exposure)** — "
    "foundational acts that attract citations faster than their age alone explains"
))
display(_top_rate)


alt.VConcatChart(...)

**Top 10 most-cited acts (raw in-degree)**

,Title,Year,Type,In-degree,Citations/yr
0,Zakon o Vladi Republike Hrvatske,2011,ZAKON,1468,97.87
1,Zakon o lokalnim porezima,2016,ZAKON,443,44.30
2,Zakon o sustavu državne uprave,2011,ZAKON,420,28.00
3,Zakon o porezu na dohodak,2016,ZAKON,276,27.60
4,Zakon o zdravstvenoj zaštiti,2008,ZAKON,209,11.61
5,Zakon o lokalnoj i područnoj (regionalnoj) sam...,2001,ZAKON,209,8.36
6,Zakon o sustavu državne uprave,2019,ZAKON,163,23.29
7,Zakon o Hrvatskoj narodnoj banci,2008,ZAKON,159,8.83
8,Zakon o poljoprivredi,2018,ZAKON,157,19.62
9,Zakon o morskom ribarstvu,2017,ZAKON,132,14.67


**Top 10 acts by citation rate (in-degree / years of exposure)** — foundational acts that attract citations faster than their age alone explains

,Title,Year,Type,In-degree,Citations/yr
0,Zakon o Vladi Republike Hrvatske,2011,ZAKON,1468,97.87
1,Zakon o lokalnim porezima,2016,ZAKON,443,44.30
2,Zakon o sustavu državne uprave,2011,ZAKON,420,28.00
3,Zakon o porezu na dohodak,2016,ZAKON,276,27.60
4,Zakon o sustavu državne uprave,2019,ZAKON,163,23.29
5,Zakon o poljoprivredi,2018,ZAKON,157,19.62
6,Zakon o morskom ribarstvu,2017,ZAKON,132,14.67
7,Zakon o zdravstvenoj zaštiti,2008,ZAKON,209,11.61
8,Zakon o mirovinskom osiguranju,2025,ZAKON,11,11.00
9,Zakon o morskom ribarstvu,2013,ZAKON,120,9.23
